# **Raw Data QA**

# All Imports Here

In [9]:
import os
glb_pth = 'd:\\GitHub\\SaaS_Product_Analysis_Based_on_Customer_Feedback_and_Competitor_Performance'
os.chdir(glb_pth)

import pandas as pd
import json
from pathlib import Path

# Importing Datasets

In [10]:
def load_product_reviews(product_path, file_name, product_name=None):
    """
    Read all raw review JSON files for one product,
    extract review-level data plus file-level metadata,
    and combine everything into one DataFrame.
    """

    chunk_dfs = []

    for json_file in product_path.rglob(file_name):

        with open(json_file, "r", encoding="utf-8") as f:
            raw_data = json.load(f)

        # review-level data
        reviews = raw_data.get("reviews", [])

        df = pd.json_normalize(reviews)

        # file-level metadata
        df["appId"] = raw_data.get("appId")
        df["reviewCount"] = raw_data.get("reviewCount")
        df["pagesRequested"] = raw_data.get("pagesRequested")
        df["scrapedAt"] = raw_data.get("scrapedAt")

        if product_name is not None:
            df["product"] = product_name

        chunk_dfs.append(df)

    if not chunk_dfs:
        return pd.DataFrame()

    return pd.concat(chunk_dfs, ignore_index=True)

In [11]:
base_path = Path(glb_pth) / "data"

zoom_df = load_product_reviews(
    base_path / "zoom",
    "zoom_raw_reviews.json",
    product_name="Zoom"
)

meet_df = load_product_reviews(
    base_path / "google_meet",
    "meet_raw_reviews.json",
    product_name="Google Meet"
)

webex_df = load_product_reviews(
    base_path / "cisco_webex",
    "webex_raw_reviews.json",
    product_name="Cisco Webex"
)

teams_df = load_product_reviews(
    base_path / "microsoft_teams",
    "teams_raw_reviews.json",
    product_name="Microsoft Teams"
)

# Inspecting Duplicate Reviews

In [26]:
zoom_df.duplicated().sum()

np.int64(0)

# Inspecting Null Values

In [28]:
webex_df.isnull().sum()


reviewId            0
rating              0
reviewer            0
date                0
reviewedIn          0
body                0
userImage           0
position            0
helpfulCounts       0
appId               0
appVersion        861
timestamp           0
language            0
reviewCount         0
pagesRequested      0
scrapedAt           0
product             0
dtype: int64

# Inspecting Review Volume

In [37]:
print(f"Review volume of Zoom: {len(zoom_df)}")
print(f"Review volume of Google Meet: {len(meet_df)}")
print(f"Review volume of Microsoft Teams: {len(teams_df)}")
print(f"Review volume of Cisco Webex: {len(webex_df)}")

Review volume of Zoom: 6000
Review volume of Google Meet: 6000
Review volume of Microsoft Teams: 6000
Review volume of Cisco Webex: 6000


# Inspecting Date Ranges

In [39]:
print(f"Date Range of Zoom: \n{min(zoom_df['date'])}   to   {max(zoom_df['date'])}")
print(f"\nDate Range of Google Meet: \n{min(meet_df['date'])}   to   {max(meet_df['date'])}")
print(f"\nDate Range of Microsoft Teams: \n{min(teams_df['date'])}   to   {max(teams_df['date'])}")
print(f"\nDate Range of Cisco Webex: \n{min(webex_df['date'])}   to   {max(webex_df['date'])}")

Date Range of Zoom: 
2026-02-06   to   2026-08-26

Date Range of Google Meet: 
2026-03-24   to   2026-08-26

Date Range of Microsoft Teams: 
2026-04-13   to   2026-08-26

Date Range of Cisco Webex: 
2024-07-20   to   2026-08-25
